In [1]:
from pyscf import gto, scf
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.operators.tensor_ordering import to_physicist_ordering, to_chemist_ordering
import numpy as np


# Define molecule, pick a basis, and run Hartree Fock

In [2]:

mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g')
mf = scf.RHF(mol).run()
print(f"Hartree Fock Energy: {mf.e_tot}")

converged SCF energy = -1.11675930739643
Hartree Fock Energy: -1.1167593073964253


# Get 1- and 2-body integrals in MO basis

In [3]:

# Transform integrals to Molecular Orbital basis
C = mf.mo_coeff
h1e_mo = C.T @ mf.get_hcore() @ C
eri_mo = np.einsum('pqrs,pi,qj,rk,sl->ijkl', mol.intor('int2e'), C, C, C, C)

# System parameters
num_spatial = mol.nao_nr()
num_alpha = (mol.nelectron + mol.spin) // 2
num_beta  = (mol.nelectron - mol.spin) // 2
nuc_repulsion = mol.energy_nuc()

# Build second-quantized Hamiltonian directly from tensors
hamiltonian = ElectronicEnergy.from_raw_integrals(h1_a=h1e_mo, h2_aa=to_physicist_ordering(eri_mo))


# Transform the Hamiltonian from 1- and 2-body integral tensors to Pauli string representation

In [4]:

# Map to qubits
mapper = JordanWignerMapper()
qubit_op = mapper.map(hamiltonian.second_q_op())

print("Jordan-Wigner Mapped Hamiltonian (Pauli Strings):")
print(qubit_op)
print(f"Number of Pauli terms: {len(qubit_op)}")


Jordan-Wigner Mapped Hamiltonian (Pauli Strings):
SparsePauliOp(['IIII', 'IIZI', 'IIIZ', 'IIZZ', 'IZII', 'IZIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'ZIII', 'ZIIZ', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.81217061+0.j, -0.22343154+0.j,  0.17141283+0.j,  0.12062523+0.j,
  0.17141283+0.j,  0.16868898+0.j,  0.04530262+0.j,  0.04530262+0.j,
  0.04530262+0.j,  0.04530262+0.j, -0.22343154+0.j,  0.16592785+0.j,
  0.16592785+0.j,  0.17441288+0.j,  0.12062523+0.j])
Number of Pauli terms: 15


# Prepare the Hartree Fock state on a quantum computer

In [5]:
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock

circ = HartreeFock(num_particles=(1,1),num_spatial_orbitals=2,qubit_mapper=mapper)
circ.draw()

┌───┐
q_0: ┤ X ├
     └───┘
q_1: ─────
     ┌───┐
q_2: ┤ X ├
     └───┘
q_3: ─────

# Estimate the expectation value of the Energy observable for the Hartree Fock state
This should match the Hartree Fock energy computed earlier!

In [9]:
from qiskit.primitives import StatevectorEstimator
estimator = StatevectorEstimator()
# result = estimator.run(circuits=circ, observables=qubit_op).result()


job = estimator.run([(circ,qubit_op)])
result = job.result()[0]
ev = result.data['evs']

print(f"Expectation value of Energy observable: {ev + nuc_repulsion}")
print(f"Hartree Fock Energy computed by PySCF:  {mf.e_tot}")

Expectation value of Energy observable: -1.1167593073964255
Hartree Fock Energy computed by PySCF:  -1.1167593073964253
